# Module 02 - Embeddings and Vector Search

**Duration:** 60 minutes

In Module 01 we used embeddings without looking at them closely.
Here we examine what they actually are, how different models compare,
what similarity metrics mean, and how ChromaDB stores and queries them at scale.

---


## 2.1 What is an embedding?

An embedding is a dense vector representation of some input (text, image, audio).
It is produced by a neural network trained to place semantically similar inputs
close together in vector space.

For text, two sentences that mean the same thing should have embeddings that point
in roughly the same direction, even if they share no words in common.

The vectors are typically 384 to 1536 numbers long, depending on the model.
Those numbers are not individually interpretable. The meaning is encoded in
the *relative positions* of vectors, not in any single dimension.

### How are embedding models trained?

Text embedding models are usually trained with **contrastive learning**.
The training signal is: "these two sentences are semantically equivalent
(e.g. a question and its answer), push their embeddings closer together;
these two sentences are unrelated, push them apart."

The most common training data is pairs of (question, answer) from the web,
or paraphrases generated from existing NLP datasets.
This is why **multi-qa** models tend to outperform **all-** models for
retrieval tasks: they were trained on question-answer pairs, which is exactly
the query/document relationship in RAG.

### The embedding space

Once trained, the model has learned a geometry where:
- Semantic similarity ≈ small angular distance between vectors
- Unrelated concepts ≈ large angular distance
- Analogies sometimes appear as parallel directions (king − man + woman ≈ queen)

This geometry is what makes search possible: instead of matching keywords,
we find nearby points in a space where "nearness" means "related meaning".

### Token limit: why it matters for RAG

Every embedding model has a maximum input length (its **token limit**):

| Model | Dimensions | Token limit |
|-------|-----------|-------------|
| all-MiniLM-L6-v2 | 384 | 256 tokens (~192 words) |
| multi-qa-mpnet-base-cos-v1 | 768 | 512 tokens (~384 words) |
| text-embedding-3-small (OpenAI) | 1536 | 8192 tokens (~6000 words) |

Text beyond the token limit is **silently truncated**: the model just
ignores everything after the cutoff. This is one reason why chunking matters:
if your chunk is 600 words and the model's limit is 384, the second half of
your chunk is invisible to the embedder.


In [ ]:
import warnings

from tqdm import TqdmExperimentalWarning

warnings.filterwarnings('ignore', category=TqdmExperimentalWarning)

import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    'The cat sat on the mat.',
    'A feline rested on a rug.',       # same meaning, different words
    'The dog chased the ball.',         # different topic
    'Quantum mechanics is very hard.',  # completely unrelated
]

embeddings = model.encode(sentences)
print('Embedding shape:', embeddings.shape)
print('\nFirst few values of sentence 0:')
print(embeddings[0][:8].round(4))


In [ ]:
# Cosine similarity between each pair
from numpy.linalg import norm

def cos_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

print('Similarity matrix:\n')
print(f'{"":43}', '       '.join([f'S{i}' for i in range(len(sentences))]))
for i, s in enumerate(sentences):
    sims = [f'{cos_sim(embeddings[i], embeddings[j]):+.3f}' for j in range(len(sentences))]
    print(f'S{i} {s[:38]:38}', '   '.join(sims))


S0 and S1 should be very similar despite sharing no words.
S0 and S2 will be lower. S0 and S3 should be the lowest.

That is semantic similarity at work.


## 2.2 Similarity metrics

Three metrics come up constantly in RAG systems.

**Cosine similarity** measures the *angle* between two vectors, ignoring their magnitude.
It ranges from -1 to 1, where 1 means identical direction.
This is the most common choice for text because sentence transformers are trained to
produce unit vectors, so magnitude does not carry meaning.

**Euclidean distance (L2)** measures the straight-line distance between two points.
Lower is more similar. Unlike cosine similarity, it is sensitive to the length of the vectors.

**Dot product** is the sum of element-wise products. For unit vectors it equals cosine similarity.
Some models (especially OpenAI's) are trained to maximise dot product rather than cosine similarity.

### Which one to use?

Always check the model card. The choice is determined by how the model was trained,
not by personal preference. Using the wrong metric can badly degrade retrieval quality.

- Models trained with cosine loss → use cosine similarity (or inner product on normalised vectors)
- Models trained with dot product loss → use dot product

In ChromaDB, you set this at collection creation time with `metadata={'hnsw:space': 'cosine'}`.

### Similarity thresholds

A cosine similarity score is not an absolute measure of relevance, it is relative
to the model and the corpus. In practice:

| Score range | Rough interpretation |
|-------------|----------------------|
| > 0.85 | Nearly identical meaning |
| 0.6 – 0.85 | Clearly related topic |
| 0.4 – 0.6 | Loosely related |
| < 0.4 | Probably not relevant |

These thresholds vary by model and domain. Always calibrate on your own data.
The `sim_th` parameter in `RAGTool` uses 0.3 as a conservative default.


In [ ]:
a = embeddings[0]  # 'The cat sat on the mat.'
b = embeddings[1]  # 'A feline rested on a rug.'
c = embeddings[3]  # 'Quantum mechanics is very hard.'

print('a vs b (related):')
print(f'  Cosine similarity:  {cos_sim(a, b):.4f}')
print(f'  Euclidean distance: {norm(a - b):.4f}')
print(f'  Dot product:        {np.dot(a, b):.4f}')

print('\na vs c (unrelated):')
print(f'  Cosine similarity:  {cos_sim(a, c):.4f}')
print(f'  Euclidean distance: {norm(a - c):.4f}')
print(f'  Dot product:        {np.dot(a, c):.4f}')


## 2.3 Comparing embedding models

Different models produce different quality embeddings for different tasks.
The choice of model is one of the most important decisions in a RAG system, more impactful than most of the "advanced" techniques in Module 05.

### Key dimensions to compare

| Dimension | Trade-off |
|-----------|-----------|
| **Dimensions** | Higher = more expressive, slower, more memory |
| **Token limit** | Higher = fewer chunking constraints |
| **Task specialisation** | QA-optimised vs general-purpose |
| **Language coverage** | Some models only work in English |
| **Speed** | Smaller models are much faster to embed large corpora |

### How to choose

1. Check the [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard), the
   standard benchmark for retrieval quality across many tasks and languages.
2. Filter by the "Retrieval" task type (not overall score).
3. Consider your language(s) and whether a multilingual model is needed.
4. Run a quick benchmark on your own documents before committing.

### A note on re-embedding

When you change the embedding model, **you must re-embed your entire corpus**
and rebuild the vector store from scratch. You cannot mix vectors from different models
in the same collection. They live in completely different spaces and mixing them
produces nonsense similarity scores.

This is another reason to benchmark early: switching models after you have ingested
100,000 documents is expensive.


In [ ]:
# Compare two models on the same queries
model_a = SentenceTransformer('all-MiniLM-L6-v2')       # general purpose, fast
model_b = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')  # optimised for QA

corpus = [
    'The AI Service Center Berlin-Brandenburg offers workshops, consulting, and compute resources.',
    'Pizza in Rome is famous for its thin crust, fresh ingredients, and wood-fired ovens.',
    'Global warming threatens ecosystems and wildlife across the planet.',
]

query = 'Where can I learn about artificial intelligence in Berlin?'

for name, m in [('all-MiniLM-L6-v2', model_a), ('multi-qa-MiniLM-L6-cos-v1', model_b)]:
    embs = m.encode(corpus)
    q_emb = m.encode(query)
    sims = [cos_sim(q_emb, e) for e in embs]
    print(f'\nModel: {name}')
    for s, sim in zip(corpus, sims):
        print(f'  {sim:.3f}  {s[:60]}')


**Exercise:** Try a query that involves a topic not directly mentioned in the corpus
(e.g. 'climate policy'). Does the more specialised model do better or worse?


## 2.4 ChromaDB

So far we have been doing brute-force search: compute similarity against every vector.
That works for a few hundred documents but is too slow for millions.

ChromaDB is a vector database that handles embedding storage, indexing, and retrieval.
It uses **HNSW** (Hierarchical Navigable Small World) by default, an approximate nearest
neighbour algorithm that finds very good results in a fraction of the time.

### How HNSW works (conceptually)

HNSW builds a multi-layer graph. The top layers have sparse long-range connections
(like a highway network); the bottom layers have dense short-range connections
(like local streets). Searching starts at the top layer and navigates down,
narrowing in on the nearest neighbours.

The trade-off: HNSW is *approximate* so it might miss the true nearest neighbour
occasionally. In practice, with default settings, the accuracy is >95% and the
speed improvement over brute force is enormous (seconds → milliseconds for large corpora).

### ChromaDB key concepts

| Concept | Description |
|---------|-------------|
| **Client** | The entry point. `chromadb.Client()` = in-memory; `PersistentClient(path=...)` = on disk |
| **Collection** | A named group of documents + their embeddings + metadata |
| **Embedding function** | Attached at creation; automatically embeds text you add |
| **Metadata** | Dict stored alongside each document; filterable with `where=` |
| **IDs** | Required; unique string identifier for each document |

We will use an in-memory client here. Module 03 switches to a persistent one.


In [ ]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()  # in-memory

embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='multi-qa-MiniLM-L6-cos-v1'
)

collection = client.create_collection(
    name='demo',
    embedding_function=embedding_func,
    metadata={'hnsw:space': 'cosine'},
)

topics = ['business', 'food', 'climate', 'technology', 'travel',
          'health', 'history', 'physics', 'music', 'AI']

texts = [
    'The AI Service Center Berlin-Brandenburg offers workshops, consulting, and compute resources.',
    'Pizza in Rome is famous for its thin crust, fresh ingredients, and wood-fired ovens.',
    'Global warming threatens ecosystems and wildlife across the planet.',
    'Graphics processing units have become essential for training AI models.',
    "Bali's beautiful beaches and rich culture make it a popular travel destination.",
    'Maintaining good health requires regular exercise, a balanced diet, and quality sleep.',
    'The French Revolution played a crucial role in shaping contemporary France.',
    "Newton's laws of motion transformed our understanding of physics.",
    "Django Reinhardt's jazz compositions are celebrated for their captivating melodies.",
    'Machine learning models improve through exposure to large amounts of training data.',
]

collection.add(
    documents=texts,
    ids=[f'doc_{i}' for i in range(len(texts))],
    metadatas=[{'topic': t} for t in topics],
)

print(f'Collection has {collection.count()} documents.')


In [ ]:
# Basic query
results = collection.query(
    query_texts=['Where can I learn about AI in Berlin?'],
    n_results=3,
)

for doc, dist, meta in zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0],
):
    similarity = round(1 - dist, 3)
    print(f'sim={similarity}  topic={meta["topic"]}  {doc[:70]}')


In [ ]:
# Exercise 2: metadata filtering
# Restrict results to a specific topic using ChromaDB's where filter

results_filtered = collection.query(
    query_texts=['environmental issues'],
    n_results=3,
    where={'topic': 'climate'},  # only return documents tagged 'climate'
)

print('Filtered results (topic=climate):')
for doc in results_filtered['documents'][0]:
    print(' -', doc)

# Try changing the filter to a different topic and see what you get


In [ ]:
# Exercise 3: visualise the embedding space with UMAP
# If umap-learn is not installed: uv pip install umap-learn matplotlib

try:
    import matplotlib.pyplot as plt
    import umap

    model_vis = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')
    embs = model_vis.encode(texts)

    reducer = umap.UMAP(n_components=2, random_state=42)
    coords = reducer.fit_transform(embs)

    plt.figure(figsize=(8, 6))
    for i, (x, y) in enumerate(coords):
        plt.scatter(x, y, s=60)
        plt.annotate(topics[i], (x, y), fontsize=8, ha='right')
    plt.title('Document embeddings (UMAP 2D projection)')
    plt.tight_layout()
    plt.show()

except ImportError:
    print('umap-learn not installed. Run: uv pip install umap-learn matplotlib')


---

**Exercises**

1. Add five new documents on topics of your choice and re-run the visualisation.
   Do the new documents cluster near related existing ones?

2. Run the same query with `n_results=5` instead of 3. Does the 4th or 5th result
   still seem relevant?

3. Check the ChromaDB docs for the `where_document` filter (different from `where`).
   Use it to restrict results to documents that contain a specific word.

---

**Further reading**

- Sentence Transformers model hub: https://www.sbert.net/docs/sentence_transformer/pretrained_models.html
- ChromaDB docs: https://docs.trychroma.com/
- HNSW paper: https://arxiv.org/abs/1603.09320
- Vector database comparison: https://zackproser.com/blog/vector-databases-compared
